## Conceptual definition

h describes the **fragility of team possession** at the current carrier's state.
Its hazard interpretation is relative to a discretization of that team
possession into the canonical actions

$$\mathcal A_0=\{\mathrm{carry}_{1s},\ \mathrm{pass}(z)\}.$$

The one-second carry unit is inspired by action-based EPV modeling
[@fernandez2019; @fernandez2021; @stalpaert2022]. Our loss-hazard interpretation
and the mixture below are project derivations.

The discrete index n counts canonical action opportunities across the team's
successive carriers, not tracking frames or equal intervals of clock time.
At a canonical decision state $s_n$, define

$$h_n(s_n)=P(\mathrm{possession\ loss\ during\ the\ current\ canonical\ action}
\mid s_n,\ \mathrm{team\ possession\ still\ active}).$$

This is a **discrete loss hazard on the team's canonical action sequence**.
A successful carry continues control with the same player. A successful pass
continues team possession with the receiver: **it is not a terminal event for h**.
It ends only the passer's individual control. This distinguishes the team-risk
interpretation of h from the individual terminal target of xLoss below.

At any reliable tracking frame, h averages over a learned pass/carry policy.
It links spatial pressure to possession fragility through explicit choice and
execution components. A carry has a canonical one-second forecast; a pass has
its own completion horizon. Thus h is a **decision-level probability**, not a
per-second rate or a forecast of loss by the end of the whole team possession.
Changing the carry horizon changes the estimand and its scale. The score can
be inspected retrospectively along individual controls or across successive
carriers in a team possession.

The policy and execution models describe an average player in the observed
context. They do not estimate player-specific control skill, body posture or
measured visual orientation. Local geometry/kinematics enter execution;
pass options and context enter choice. A high score indicates difficult
modeled alternatives under that policy, not inevitable failure. Training uses
the agreed operational loss labels; rare out-of-play/clearance conventions are
documented separately in the benchmark rather than redefining every individual
control ending as a turnover.

## Mathematical definition

Condition all quantities on the current state $s$. Let

* $q=P(\mathrm{pass}\mid s)$; $1-q$ is carry-choice probability.
* $d=P(\mathrm{loss\ within\ 1s}\mid s,\mathrm{carry})$.
* $\pi_j=P(\mathrm{destination}\ j\mid s,\mathrm{pass})$, with $\sum_j\pi_j=1$.
* $p_j=P(\mathrm{pass\ succeeds}\mid s,j)$.

First average pass failure over destinations:

$$f(s)=\sum_j\pi_j(1-p_j)=1-\sum_j\pi_jp_j.$$

Then apply total probability over the two action branches:

$$\boxed{h(s)=(1-q)d+qf(s).}$$

The weighted carry contribution is $(1-q)d$; the weighted pass contribution is
$qf$. These contributions add exactly to h. They differ from the branch risks
d and f themselves. A real case can have high pass risk but low h when passing
is unlikely, or vice versa. All components are evaluated from the state representation at that frame;
the observed next choice and its outcome are not supplied to this formula.
Tracking velocities use the established centered seven-frame smoother
(±0.3 s at 10 Hz), so this is an offline tracking analysis rather than a strictly
causal real-time estimator. No realized future route is supplied as a predictor.

The destination model uses **raw pitch control as an input feature**, while
$p_j$ is the **calibrated pass-success probability** from the physical control
model and its isotonic mapping. The implementation calls
$\sum_j\pi_jp_j$ `rppcf`: this is the project's transition-weighted pass-success
aggregate. Spearman names the underlying control field PPCF; this does not
establish `rPPCF` as a universal term [@spearman2017; @spearman2018beyond].

This composition and its pressure interpretation are **own derivation**.
Discrete hazard terminology follows the event/decision-level idea of a
conditional failure probability [@singer2003; @suresh2022discrete]. The internal physical tackle
hazard is a rate and can exceed one; h is a probability in $[0,1]$.

The component vignettes give their assumptions, features, training and references:
[carry loss](../01_ball_drive_risk/01_ball_drive_risk.ipynb),
[pass probability](../02_pass_probability/02_pass_probability.ipynb),
[destination selection](../03_transition_model/03_transition_model.ipynb), and
[action selection](../04_action_selection/04_action_selection.ipynb).


## Absolute bands and equivalent constant-risk half-life

As a reference scale, consider repeated canonical team actions with the same
conditional loss probability h. The probability of avoiding loss over n such
actions is $(1-h)^n$; successful passes continue this sequence. This is not a
forecast of the actual team's remaining possession length, because future
states and their risks change. The **equivalent constant-risk half-life** is
the number of equal-risk actions that halves this reference no-loss probability:

$$n_{1/2}=\frac{\log(1/2)}{\log(1-h)},\qquad
h_n=1-2^{-1/n}.$$

We use the cutoffs for n=8,4,2. This is an interpretable **project design
choice**, not thresholds learned from outcomes or published clinical-style
pressure categories. The API preserves the established English labels:

| Band | h interval | Constant-risk half-life interpretation |
|---|---|---|
| None | $0\le h<0.082996$ | More than 8 decisions; infinite at h=0 |
| Low | $0.082996\le h<0.159104$ | $4<n_{1/2}\le8$ decisions |
| Medium | $0.159104\le h<0.292893$ | $2<n_{1/2}\le4$ decisions |
| High | $0.292893\le h\le1$ | At most 2 decisions |

The code uses the exact formula, not the rounded displayed numbers. Values
on a threshold enter the higher band. The theoretical n=1 cutoff is h=0.5;
it remains within High and does not define a fifth operational band.
**None means the lowest modeled risk band, not absence of defensive pressure.**
Half-life here is a scale interpretation in **decisions, not seconds**. Passes
have variable duration and actual future states/risks will change; this does
not forecast how many seconds the team will retain possession.

## Transparent numerical example

Suppose q=0.4, d=0.1, destination probabilities are (0.25,0.5,0.25), and pass
success probabilities are (0.7,0.4,0.2). Expected pass failure is f=0.575,
so h=0.6×0.1+0.4×0.575=0.29, in Medium. These are explicitly synthetic inputs.


In [4]:
import numpy as np
from PRESS.utils.hazard import compute_hazard_raw
from PRESS.utils.possession import hazard_band_thresholds, hazard_to_band
from scripts.build_h_vignette_figures import main as rebuild_h_figures

transition = np.array([0.25, 0.50, 0.25])
pass_success = np.array([0.70, 0.40, 0.20])
h, drive_loss, pass_failure, weighted_pass_success = compute_hazard_raw(
    0.1, transition, pass_success, 0.4)
assert np.isclose(h, 0.29) and hazard_to_band(h) == "Medium"
print({"h": h, "carry contribution": 0.6 * drive_loss,
       "pass contribution": 0.4 * pass_failure, "band": hazard_to_band(h)})
print("Exact cutoffs:", hazard_band_thresholds())
rebuild_h_figures()


{'h': 0.29, 'carry contribution': 0.06, 'pass contribution': 0.22999999999999998, 'band': 'Medium'}
Exact cutoffs: {1: 0.5, 2: 0.2928932188134524, 4: 0.1591035847462855, 8: 0.08299595679532878}


## Validation of the fixed absolute bands

The following table uses **15,085 canonical held-out action states**, pooling
the five matching outer test pairs. Thresholds were fixed by the half-life
construction; they are not quantiles or boundaries selected to maximize observed
separation. The observed target is **failure of the current canonical action**,
not terminal individual loss.

| Banda | Azioni | h medio previsto | Fallimenti osservati |
|---|---:|---:|---:|
| Minima | 8.716 | 2,94% | 3,37% |
| Low | 2.984 | 11,88% | 12,06% |
| Medium | 2.534 | 21,39% | 21,23% |
| High | 851 | 36,51% | 32,08% |

Minima is the API's `None` label. The ordering is clear, and the observed rates
fall inside their corresponding absolute bands. Low and Medium closely match
their mean predictions. High is somewhat pessimistic: 36.51% predicted versus
32.08% observed. This is evidence of useful risk separation and approximate
calibration, not proof that 8/4/2 are unique optimal cutoffs or universal football
categories. The point of the scale is its absolute probabilistic meaning.

![Observed action-level failure and mean predicted h for fixed bands.](../../figures/h_band_validation.png)

Error bars resample **whole matches**, 2,000 times with a fixed seed, keeping the
band cutoffs unchanged [@efron1993bootstrap]. They describe uncertainty within
this ten-match development sample, not external-season validation. Longer
controls supply several canonical actions; raw counts are not independent
possessions. The full [composed-risk benchmark](../05b_composed_risk_benchmark/05b_composed_risk_benchmark.ipynb)
retains CE, gain over null and the pooled ten-quantile calibration curve.

Because h marginalizes over choice, splitting the data by the **observed** pass
or carry changes conditioning: h need not equal the event rate within each
chosen-action subset. Conditional execution checks use the relevant branch
model. No additional h calibrator is fitted here.


In [6]:
from pathlib import Path
import pandas as pd
from scripts.build_pressure_report_figures import main as rebuild_pressure_figures

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").is_file())
validation = pd.read_csv(root / "data/examples/pressure_report/band_validation.csv", keep_default_na=False)
assert validation["n"].sum() == 15085 and validation["losses"].sum() == 1465
assert validation["observed_loss"].is_monotonic_increasing
print(validation[["band", "n", "mean_h", "observed_loss"]].to_string(index=False))
rebuild_pressure_figures()


  band    n   mean_h  observed_loss
  None 8716 0.029404       0.033731
   Low 2984 0.118782       0.120643
Medium 2534 0.213910       0.212313
  High  851 0.365087       0.320799
Pressure report figures rebuilt from fixed aggregate tables.


![Synthetic branch mixture and exact operational band boundaries.](../../figures/vignette_05_hazard_bands.png)

## Evaluation frames and canonical training actions

The same state formula can be queried at **any reliable tracking frame**.
Canonical action segmentation defines the reference decision units for
training and the discrete-hazard interpretation. It does not restrict state
queries to integer-second times: an extra snapshot uses the same canonical
carry/pass forecasts, rather than redefining carrying as a shorter action.
An evaluated tracking frame is not automatically a new canonical decision.

* A 0.4 s individual control ending with a pass has a pass-choice training row
  at 0.4 s. The 0 s state can be retained as an additional descriptive h snapshot.
* A 2.4 s control ending with a pass has successful canonical carries at 0 s and
  1 s, plus the pass at 2.4 s. A descriptive h profile may also include the 2 s
  snapshot; that successful 0.4 s tail is not an action-selection training row.
* If that 2.4 s control ends in direct loss, the carry starting at 2 s is a failed
  training action, even though observed failure occurs after 0.4 s. Its target
  remains loss within the canonical one-second forecast.
* A pass before one complete second retains the possession-start context as
  previous action. After at least one complete second of carrying, previous
  action is ball drive. Additional snapshots do not create training labels.

Canonical h benchmarking uses the verified action cohort and matching outer
models. It does not mix descriptive microtail snapshots into labeled actions.
Clearances and fouls are not extra action-choice branches; complete carries
before those endings can contribute canonical states. A foul-interrupted short
tail is not automatically labeled a failed carry. Terminal individual-loss
conventions and the whole-parent quality rules are documented in the
[separate composed h/xLoss benchmark](../05b_composed_risk_benchmark/05b_composed_risk_benchmark.ipynb).



## Context-checked real examples

Two main cases show different pressure trajectories. Every **canonical evaluated
checkpoint** has a pitch panel; this does not claim to be a dense 10 Hz profile.
The yellow star identifies the carrier, the attack direction is stated, and
velocity arrows are scaled drawing aids. Titles show h, its band, weighted
branch contributions, nearest-defender distance and the number of active
annotated on-ball engagements (OBE).

These are frozen matching-fold predictions at observed tracking states. Cases
are selected retrospectively for interpretation, not randomly or as a new
performance evaluation. Geometry, carrier movement, event origin, pass result,
pass options and OBE timing were cross-checked. Tracking and annotations support
the descriptions; body orientation, intention and causal tactical effects are
not observed, and no video verification is claimed.

## Case A: escape after recovery {#pressure-escape-case}

**Nathaniel Atkinson, Melbourne City–Macarthur, 7 March 2025.** The control starts
from a **pass interception**, not reception. An opponent engagement spans the
recovery and most of the control. At the initial state, the nearest opponent is
2.69 m away; at the final pass state, the nearest is 6.38 m away and the annotated
engagement has ended. The carry-loss branch falls from 10.10% to 0.42%, and
expected pass failure from 42.40% to 5.66%. h decreases
**36.73%→13.52%→3.45%→3.03%**.

The endpoint is a completed 5.54 m lateral pass to Kai Trewin. Net longitudinal
progress is approximately −0.54 m: the interpretation is **relief from immediate
pressure and safer options**, not a progressive dribble. Spatial separation,
branch-risk reduction and the completed release jointly support the escape
label. This extends the [SkillCorner-inspired interpretation](../08_possession_typologies/08_possession_typologies.ipynb)
to recoveries, rather than claiming to reproduce its reception-only metric.

::: {.panel-tabset}

### t = 0.0 s · High

![Nathaniel Atkinson, checkpoint 1; h=36.7%, observed current frame.](../../figures/h_case_A_checkpoint_01.png)

### t = 1.0 s · Low

![Nathaniel Atkinson, checkpoint 2; h=13.5%, observed current frame.](../../figures/h_case_A_checkpoint_02.png)

### t = 2.0 s · None

![Nathaniel Atkinson, checkpoint 3; h=3.4%, observed current frame.](../../figures/h_case_A_checkpoint_03.png)

### t = 3.4 s · None

![Nathaniel Atkinson, checkpoint 4; h=3.0%, observed current frame.](../../figures/h_case_A_checkpoint_04.png)

:::


## Case B: drawing pressure {#drawing-pressure-case}

**Harry Van Der Saag, Brisbane–Perth, 21 December 2024,04:27.1–04:30.4.** He
receives with the nearest opponent 9.14 m away and no active annotated engagement.
He advances 7.27 m along the attacking axis. At release, three engagement events
are active and the nearest opponents are 4.24 m and 4.40 m away.

h rises **1.18%→3.72%→16.41%→34.98%**, while the carry and pass branches both
become riskier. The sequence lasts 3.3 s and ends with a completed 11.05 m lateral
pass to Adam Zimarino. These observations support an operational **drawing
pressure** pattern: a relatively free reception develops into engagement with
several opponents before release [@skillcorner2026drawing]. They do not identify
a conscious baiting intention. This is a successful action under increased
modeled risk, not an assertion that High implies certain loss.

::: {.panel-tabset}

### t = 0.0 s · None

![Harry Van Der Saag, checkpoint 1; h=1.2%, observed current frame.](../../figures/h_case_B_checkpoint_01.png)

### t = 1.0 s · None

![Harry Van Der Saag, checkpoint 2; h=3.7%, observed current frame.](../../figures/h_case_B_checkpoint_02.png)

### t = 2.0 s · Medium

![Harry Van Der Saag, checkpoint 3; h=16.4%, observed current frame.](../../figures/h_case_B_checkpoint_03.png)

### t = 3.3 s · High

![Harry Van Der Saag, checkpoint 4; h=35.0%, observed current frame.](../../figures/h_case_B_checkpoint_04.png)

:::


The [third case](../08_possession_typologies/08_possession_typologies.ipynb#carrying-case)
shows Trent Ostler carrying for 6.5 s with a close marker rather than becoming
unmarked. It makes the distinction between spatial escape, pressure attraction
and retention during a sustained engagement visible.

![Three contextualized canonical trajectories and their weighted branch contributions.](../../figures/h_real_case_trajectories.png)

[Case probabilities](../../data/examples/hazard/cases.csv) ·
[event/outcome summaries](../../data/examples/hazard/case_summary.csv) ·
[checkpoint geometry and engagement counts](../../data/examples/hazard/context_signals.csv) ·
[context verification](../../data/examples/hazard/metadata.json).


In [11]:
from pathlib import Path
import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").is_file())
cases = pd.read_csv(root / "data/examples/hazard/cases.csv", dtype={"match_id": str}, keep_default_na=False)
recomputed = (1 - cases.p_pass) * cases.drive_loss + cases.p_pass * cases.pass_failure
assert np.allclose(recomputed, cases.h, atol=1e-12)
assert np.array_equal(hazard_to_band(cases.h), cases.band)
print(cases[["case", "time_since_start", "action", "p_pass", "drive_loss",
             "pass_failure", "h", "band"]].to_string(index=False))


case  time_since_start     action   p_pass  drive_loss  pass_failure        h   band
   A               0.0 ball_drive 0.824390    0.101020      0.424040 0.367314   High
   A               1.0 ball_drive 0.656374    0.075469      0.166465 0.135196    Low
   A               2.0 ball_drive 0.394807    0.001676      0.084700 0.034455   None
   A               3.4       pass 0.499338    0.004170      0.056560 0.030330   None
   B               0.0 ball_drive 0.203357    0.000240      0.056961 0.011774   None
   B               1.0 ball_drive 0.392803    0.001564      0.092375 0.037235   None
   B               2.0 ball_drive 0.467033    0.028189      0.319292 0.164144 Medium
   B               3.3       pass 0.747304    0.307685      0.364101 0.349845   High
   C               0.0 ball_drive 0.611885    0.160371      0.550528 0.399102   High
   C               1.0 ball_drive 0.357306    0.081914      0.489878 0.227682 Medium
   C               2.0 ball_drive 0.211229    0.049620      0.643

## Use the package on a selected frame

The following executable API example uses **synthetic match data**. Small grids
make it quick and change the model inputs; use the research defaults for
analysis. Initialization does not itself calculate h. Select a reliable frame,
run `compute_frame`, then retrieve its component decomposition. Use
`PRESS.from_skillcorner(...)` from the [match guide](../../usage.qmd) for
processed real matches, with possession/action context and xT supplied as there.


In [13]:
from PRESS import PRESS
from PRESS.datasets import load_demo_match

tracking, possessions, actions, xT_grid = load_demo_match()
model = PRESS(tracking, possessions_data=possessions, actions_data=actions)
possession_id = model.get_possessions().possession_id.iloc[0]
frame_id = int(model.get_frames(possession_id, valid_only=True).frame.iloc[5])
result = model.compute_frame(frame_id, xT_grid,
    ppcf_kwargs={"r_max": 12, "step": 6}, bcos_kwargs={"r_max": 8, "step": 4})
hazard = model.get_hazard_result(frame_id)["Hazard"]
print("hazard:", hazard, "band:", hazard_to_band(hazard))


hazard: 0.012514264786956159 band: None


## From snapshots to possession profiles

h can describe where an individual control becomes difficult, remains under
risk, or escapes risk. For descriptive comparisons, state the checkpoint
sampling and the conditioning population. Longer possessions contribute more
canonical rows; a count of rows in a band is not automatically seconds under
pressure. Comparing teams or players also requires exposure, position and
opponent context; raw band shares do not isolate individual skill.

The [survival/difficulty vignette](../07_survival_curve/07_survival_curve.ipynb)
computes $D_t=-\log(1-h_t)$ and cumulative difficulty. Multiplying
$\prod_t(1-h_t)$ across a supplied retrospective profile is sensitive to its
sampling. **Overlapping one-second forecasts at extra microtail or dense
tracking snapshots must not be treated as independent successive actions.**
Keep a canonical decision sequence for such summaries, or explicitly report a
descriptive index. A denser plot alone must not imply more exposure or loss
probability. There is no conversion of h into a clock-time rate here.

The [typology vignette](../08_possession_typologies/08_possession_typologies.ipynb)
uses band trajectories and observed endings for retrospective labels. The
[team report](../09_league_level_report/09_league_level_report.ipynb) illustrates
aggregation; these are project-defined analysis tools, not externally validated
player or team rankings.

## Evaluation, calibration and scope

The [separate h/xLoss benchmark](../05b_composed_risk_benchmark/05b_composed_risk_benchmark.ipynb)
reports held-out action-loss scores and the pooled calibration curve. h has no
additional post-hoc calibrator. Good calibration of individual components does
not mathematically guarantee calibration of their composition; it is assessed
on the composed predictions. These are development OOF results on ten
repeatedly studied matches, not an untouched external test set.

[Residual risk](../05a_residual_risk/05a_residual_risk.ipynb) completes the model
toward individual terminal xLoss through
$xLoss=h+(1-q)(1-d)c$. It predicts a different target: a completed pass ends the individual xLoss
sequence without loss, whereas the team sequence underlying h continues. The raw h pressure profile
and its bands remain usable without that optional continuation model.

The two-action structure excludes shots, and the benchmark is limited to the
agreed outfield, verified-endpoint cohort. Tracking and event-annotation noise,
straight short-horizon extrapolation, unobserved body posture/control skill,
average-player assumptions and limited tactical interactions remain relevant
limitations. The component and BCOS vignettes document these assumptions and
their literature sources. Band thresholds and case-based interpretation are
explicit project choices; their usefulness is not a new benchmark result.


## From h to player and team performance

The [ten-match report](../09_league_level_report/09_league_level_report.ipynb)
uses a primary **exposure-normalized retention percentage**, counting each
individual possession once if at least one observed canonical state is in
Medium or High:

$$\mathrm{Retention}_j=100\,
\frac{\#\{\mathrm{pressure\ situations\ ending\ without\ individual\ loss}\}}
{\#\{\mathrm{pressure\ situations\ faced}\}}.$$

This is distinct from action-level failure calibration and from an
observed-minus-expected action score. Counts, match coverage and uncertainty
accompany percentages. The [pattern vignette](../08_possession_typologies/08_possession_typologies.ipynb)
defines the three behaviours and their different eligibility denominators,
inspired by SkillCorner [@skillcorner2026; @skillcorner2026escape;
@skillcorner2026drawing; @skillcorner2026retained]. The report follows the
exposure/context emphasis of CIES [@poli2024pressure].

## References and project choices

* Action representation and EPV context: Fernández and Stalpaert
  [@fernandez2019; @fernandez2021; @stalpaert2022].
* Physical pass/control ingredients: Spearman [@spearman2017;
  @spearman2018beyond]; full component equations and remaining sources are in
  the linked execution-model vignettes.
* Discrete-hazard interpretation: Singer/Willett and Suresh et al.
  [@singer2003; @suresh2022discrete]. Our action-index adaptation, mixture,
  equivalent-half-life band cutoffs and report rules are project choices.
* Resampling uncertainty: Efron/Tibshirani [@efron1993bootstrap].
* Tracking and dynamic annotations: SkillCorner Open Data
  [@skillcorner-opendata]. Named pressure behaviours are conceptual references,
  not numerical comparisons with a proprietary model.
